# 3.3 SARIMA: the classical forecasting benchmark
**Purpose.** An auxiliary benchmark (paper Section 3.7): seasonal ARIMA forecasts of relative occupancy and relative flow, scored like the two forecasting approaches. SARIMA only uses each district's own series, so it enters no comparison.

**Method.** SARIMA(1,0,1)(0,1,1) with a daily season of 96 slots, the same untuned order for every district and variable (19 × 2 = 38 models). The parameters are estimated by maximum likelihood on 2022 only and then fixed. Invalid slots are missing values that the Kalman filter skips, and nothing is filled in. At each origin *t*, the state is updated with the observations up to *t*, and the warning score is the largest forecast relative value in *t*+1 … *t*+*H*.

**Implementation.** The 2022 likelihood is computed on the seasonally differenced series (statsmodels' simple differencing, about 8 times faster), so a slot drops out of the likelihood when it or the slot one day earlier is invalid. The model and its four parameters stay the same. The forecasts use the full state-space model, in one filter pass from 1 January 2022 to the end of the scored year (`madrid_congestion/sarima.py`, which the 2024 run also uses).

**Outputs.** The 38 parameter sets go to `models/sarima-2022.json`, and the 2022 and 2023 scores to `data/interim/scores-sarima.parquet`. 2024 is not read.

## 1. Setup and series

In [1]:
import time

import numpy as np
import polars as pl

from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, PANEL_YEAR, PROCESSED_DATA_DIR, VALIDATION_YEAR
from madrid_congestion.evaluation import forecast_mae, warning_table
from madrid_congestion.features import origin_rows
from madrid_congestion.frozen import check_design, save_frozen
from madrid_congestion.labels import add_labels, references, studied_districts, year_grid
from madrid_congestion.sarima import (
    VARIABLES,
    fit_and_forecast_all,
    fit_sarima,
    forecast_paths,
    sarima,
    sarima_grid,
    sarima_scores,
    series_values,
)

STEPS = max(HORIZONS.values())  # 4 slots: the 60-minute window; the 30-minute score uses the first 2
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, VALIDATION_YEAR)}
districts = studied_districts(series[PANEL_YEAR])
check_design(districts)  # the code still builds the frozen design, SARIMA order included
refs = references(year_grid(series[PANEL_YEAR], districts, PANEL_YEAR))
labels = {year: add_labels(year_grid(df, districts, year), refs) for year, df in series.items()}

# One continuous 15-minute grid per district, 2022 then 2023; relative values of valid slots, NaN otherwise
grid = sarima_grid([labels[PANEL_YEAR], labels[VALIDATION_YEAR]])
values = {(district, variable): series_values(grid, district, variable) for district in districts for variable in VARIABLES}
fit_slots = labels[PANEL_YEAR].filter(pl.col("distrito") == districts[0]).height  # the 2022 slots of each series
print(len(values), "series |", fit_slots, "slots of 2022 per series |", len(values[districts[0], "occupancy"]), "slots in all")

38 series | 35040 slots of 2022 per series | 70080 slots in all


## 2. Check on a small subset
Before the full run, on one district and variable: a timed fit on January 2022, and the forecast paths over January 2022 and the first week of 2023, compared with statsmodels' own forecasts at a few origins. These numbers are a check, not results.

In [2]:
small = values[districts[0], "occupancy"]
january = small[: 31 * 96]
start = time.time()
params = fit_sarima(january)
print(f"fit on January 2022: {time.time() - start:.1f} s |", {name: round(value, 4) for name, value in params.items()})

check = np.r_[january, small[fit_slots : fit_slots + 7 * 96]]  # January 2022, then the first week of 2023
start = time.time()
paths = forecast_paths(check, params, STEPS)
print(f"forecast paths for {len(check)} slots: {time.time() - start:.1f} s")
for origin in (len(january) + 50, len(check) - 10):
    own = sarima(check[: origin + 1]).filter(np.array(list(params.values()))).forecast(STEPS)
    assert np.allclose(paths[origin], own, atol=1e-10), origin
print("forecast paths equal statsmodels' own forecasts")

fit on January 2022: 57.2 s | {'ar.L1': 0.9707, 'ma.L1': -0.173, 'ma.S.L96': -0.9985, 'sigma2': 0.0014}
forecast paths for 3648 slots: 1.9 s
forecast paths equal statsmodels' own forecasts


## 3. The 38 models
Each district and variable is fitted on 2022 and filtered over 2022 and 2023 with the fixed parameters (`fit_and_forecast_all`, run in parallel).

In [3]:
start = time.time()
fitted = fit_and_forecast_all(values, fit_slots, STEPS)  # (district, variable) -> (2022 parameters, forecast paths)
print(f"38 models in {(time.time() - start) / 60:.0f} minutes")
parameters = pl.DataFrame([{"distrito": district, "variable": variable, **params} for (district, variable), (params, _) in fitted.items()])
parameters.with_columns(pl.exclude("distrito", "variable").round(4))

38 models in 90 minutes


distrito,variable,ar.L1,ma.L1,ma.S.L96,sigma2
i64,str,f64,f64,f64,f64
1,"""occupancy""",0.9701,-0.1419,-0.9356,0.0019
1,"""flow""",0.9803,0.1018,-0.9628,0.0007
2,"""occupancy""",0.9559,0.0219,-0.9165,0.003
2,"""flow""",0.9779,0.1155,-0.9448,0.0008
3,"""occupancy""",0.9621,-0.1865,-0.9541,0.0034
3,"""flow""",0.9652,-0.3994,-0.9787,0.0043
4,"""occupancy""",0.9748,-0.044,-0.946,0.0019
4,"""flow""",0.9842,0.1408,-0.9685,0.0006
5,"""occupancy""",0.9653,0.1873,-0.9479,0.0021


## 4. Scores on the usable calm origins
The largest forecast in *t*+1 … *t*+*H* of the district's relative occupancy or flow (`sarima_scores`), on the same usable calm origins as every other score.

In [4]:
paths = {key: path for key, (_, path) in fitted.items()}
parts, mae = [], []  # one table of scores per year and horizon
for year, df in labels.items():
    for minutes in HORIZONS:
        rows = origin_rows(df, districts, minutes)
        year_scores = sarima_scores(grid, paths, rows, minutes)
        parts.append(pl.concat(
            [rows.select(pl.lit(year).alias("year"), pl.lit(minutes).alias("minutes"), "distrito", "slot_start", target=pl.col(f"target_{minutes}")), year_scores],
            how="horizontal",
        ))
        for variable, reference in (("occupancy", "threshold"), ("flow", "flow_reference")):
            mae.append({"minutes": minutes, "year": year, "forecast": variable,
                        **forecast_mae(year_scores[f"sarima_{variable}"], rows[f"{variable}_max_{minutes}"], rows[f"{variable}_lag0"], rows[reference])})
scores = pl.concat(parts)
ap = []
for (year, minutes), df in scores.group_by("year", "minutes"):
    table = warning_table(df["target"], {f"sarima_{variable}": df[f"sarima_{variable}"] for variable in VARIABLES})
    ap.extend({"minutes": minutes, "year": year, "base_rate": df["target"].mean(), "score": row["score"], "ap": row["ap"]} for row in table.iter_rows(named=True))
pl.DataFrame(ap).sort("minutes", "score", "year").with_columns(pl.col("base_rate", "ap").round(4))

minutes,year,base_rate,score,ap
i64,i64,f64,str,f64
30,2022,0.017,"""sarima_flow""",0.4438
30,2023,0.0174,"""sarima_flow""",0.4207
30,2022,0.017,"""sarima_occupancy""",0.6038
30,2023,0.0174,"""sarima_occupancy""",0.5801
60,2022,0.0342,"""sarima_flow""",0.5145
60,2023,0.0352,"""sarima_flow""",0.4905
60,2022,0.0342,"""sarima_occupancy""",0.6467
60,2023,0.0352,"""sarima_occupancy""",0.6362


**Forecast accuracy.** MAE of the SARIMA window maximum against the realised one, next to the naive forecast. 2022 is the fitting year.

In [5]:
pl.DataFrame(mae).sort("minutes", "forecast", "year").with_columns(pl.exclude("minutes", "year", "forecast").round(4))

minutes,year,forecast,mae_relative,naive_mae_relative,mae_original_units,naive_mae_original_units,mae_reduction
i64,i64,str,f64,f64,f64,f64,f64
30,2022,"""flow""",0.027,0.0395,16.9542,24.6971,0.3156
30,2023,"""flow""",0.0255,0.0386,15.9436,24.1034,0.3388
30,2022,"""occupancy""",0.0401,0.0508,0.4952,0.6265,0.2101
30,2023,"""occupancy""",0.0402,0.0511,0.4993,0.635,0.2144
60,2022,"""flow""",0.0362,0.0587,22.5273,36.295,0.3828
60,2023,"""flow""",0.0349,0.0579,21.6657,35.8071,0.3973
60,2022,"""occupancy""",0.0529,0.0751,0.648,0.9202,0.2962
60,2023,"""occupancy""",0.0531,0.0758,0.656,0.9343,0.2989


## 5. Frozen outputs

In [6]:
save_frozen("sarima-2022.json", {variable: {str(district): fitted[district, variable][0] for district in districts} for variable in VARIABLES})
scores.write_parquet(INTERIM_DATA_DIR / "scores-sarima.parquet")
print(scores.group_by("year", "minutes").len().sort("year", "minutes"))

shape: (4, 3)
┌──────┬─────────┬────────┐
│ year ┆ minutes ┆ len    │
│ ---  ┆ ---     ┆ ---    │
│ i32  ┆ i32     ┆ u32    │
╞══════╪═════════╪════════╡
│ 2022 ┆ 30      ┆ 475434 │
│ 2022 ┆ 60      ┆ 453544 │
│ 2023 ┆ 30      ┆ 482391 │
│ 2023 ┆ 60      ┆ 461884 │
└──────┴─────────┴────────┘


**Result (2023 validation, not test results).** The 38 models were fitted on 2022 in about 90 minutes. The autoregressive parameter lies between 0.942 and 0.984, and the seasonal moving-average parameter between −0.998 and −0.905. The value of −0.998 (flow, district 5) is close to the invertibility boundary. On 2023, the SARIMA occupancy score has an AP of 0.580 at 30 minutes and 0.636 at 60 minutes, and the SARIMA flow score 0.421 and 0.491 (base rates 1.74% and 3.52%). SARIMA reduces the naive MAE by 21% and 30% for occupancy and by 34% and 40% for flow.

**Conclusion.** The SARIMA parameters are fixed. SARIMA is an auxiliary benchmark and enters no comparison.